In [1]:
import pandas as pd
import os
import json
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from tqdm import tqdm
import datetime

In [41]:
os.chdir('/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis')

In [42]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'

In [44]:
nace_description_path = "data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv"
nace_descriptions = pd.read_csv(nace_description_path, sep="\t")

In [2]:
def get_zero_shot_user_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["user_prompt_zero_shot"]

def get_few_shot_user_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["user_prompt_few_shot"]

def get_system_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["system_prompt"]

In [3]:
user_prompt_topic = """
TASK
For the following business sector, create short descriptions of realistic business models for exisiting companies. 

\nDEFINITION
\n{includes} {includes_also}
\n
\n{excludes}
\n
\nHere are some possible subsections: {subsections}

INSTRUCTION
- Create a list of {num_samples} different realistic explanations of a business model 
- The length should be one sentence
- Pick one or multiple subsections for this business model
- Do not explain what you did or what you used
"""

In [47]:
def generate_topic(
        num_samples: int, 
        includes: str,
        includes_also: str, 
        excludes: str,
        subsections: list,
        prompt_path: str,  
        temperature: float = 0.4, 
        model: str = "gpt-4o-mini"
): 

    # Initialize LLM
    if model == "gpt-4o-mini":
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature
        )
    elif model == "openai/gpt-oss-120b":
        llm = ChatOllama(
            model=model,
            temperature=temperature, 
            base_url="http://10.80.20.127:11434/"
        )


    prompt = ChatPromptTemplate.from_messages([
        ("system", ""),
        ("human", user_prompt_topic)
    ])

    # Subsections string
    subsections_str = "\n - " + "\n - ".join(subsections)

    # Adapt excludes
    if excludes != "": 
        excludes = "Excludes: " + excludes

    # Chain
    chain = prompt | llm

    # inputs
    input = {
        "num_samples": num_samples,
        "includes": includes,
        "includes_also": includes_also,
        "excludes": excludes,
        "subsections": subsections_str,
        }

    formatted_prompt = prompt.invoke(input)

    #print("Formatted Prompt:", formatted_prompt)

    # Run
    response = chain.invoke(input)

    #print(response.content)

    return formatted_prompt, response.content

In [48]:
def generate_synthetic_data(
        num_samples: int, 
        gold_standard: list, 
        includes: str,
        includes_also: str, 
        excludes: str,
        subsections: list,
        prompt_path: str,  
        temperature: float = 0.4, 
        model: str = "gpt-4o-mini", 
        topic: str = None
): 

    # Initialize LLM
    if model == "gpt-4o-mini":
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature
        )
    elif model == "openai/gpt-oss-120b":
        llm = ChatOllama(
            model=model,
            temperature=temperature, 
            base_url="http://10.80.20.127:11434/"
        )

    # Prompt
    if gold_standard == []: 
        #print("Zero Shot!")
        prompt = ChatPromptTemplate.from_messages([
            ("system", get_system_prompt(prompt_path)),
            ("human", get_zero_shot_user_prompt(prompt_path))
        ])
        gold_standard_str = ""
        
    else: 
        prompt = ChatPromptTemplate.from_messages([
            ("system", get_system_prompt(prompt_path)),
            ("human", get_few_shot_user_prompt(prompt_path))
        ])
        gold_standard_str = ""
        for i, text in enumerate(gold_standard): 
            gold_standard_str += f"Example {i+1}:\n{text}\n\n"
        gold_standard_str = gold_standard_str[:-2]

    # Subsections string
    subsections_str = "\n - " + "\n - ".join(subsections)

    # Adapt excludes
    if excludes != "": 
        excludes = "Excludes: " + excludes

    # Chain
    chain = prompt | llm

    # inputs
    input = {
        "num_samples": num_samples,
        "gold_standard": gold_standard_str,
        "includes": includes,
        "includes_also": includes_also,
        "excludes": excludes,
        "subsections": subsections_str,
        }
    print(prompt)
    # is topic used? topic in params -> business model must be in user prompt
    if topic is not None:
        if "business_model" not in prompt.input_variables: 
            print("If topic is given, prompt must conatin '{business_model}'")
            return False
        else: 
            input["business_model"] = topic

    formatted_prompt = prompt.invoke(input)

    #print("Formatted Prompt:", formatted_prompt)

    # Run
    response = chain.invoke(input)

    #print(response.content)

    return formatted_prompt, response.content

In [49]:
def split_synthetic_data(content: str, num_samples: int): 
    content_list = content.split("\n")
    content_list = [c for c in content_list if c != ""]
    # if len(content_list) != num_samples: 
    #     print("Warning: length of creates examples != num_samples!")
    return content_list

In [50]:
def get_sublevels(nace_class, level): 
    nace_class_temp = nace_class
    nace_id = nace_descriptions[nace_descriptions["CODE"] == nace_class_temp]["ID"].iloc[0]
    nace_class_lvl_2 = []
    nace_class_lvl_3 = []
    nace_class_lvl_4 = []

    for _, row in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id].iterrows(): 
        nace_id_temp = row["ID"]
        nace_class_lvl_2.append(f'{row["NAME"]}')
        for _, row_2 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp].iterrows(): 
            nace_id_temp_temp = row_2["ID"]
            nace_class_lvl_3.append(f'{row["NAME"]}: {row_2["NAME"]}')
            for _, row_3 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp_temp].iterrows(): 
                nace_class_lvl_4.append(f'{row["NAME"]}: {row_2["NAME"]}: {row_3["NAME"]}')
        
    if level == 2: 
        return nace_class_lvl_2
    if level == 3: 
        return nace_class_lvl_3
    if level == 4: 
        return nace_class_lvl_4

In [51]:
generate_nace_class = "A"

includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
assert includes is not None and includes != ""
includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
includes_also = "" if pd.isna(includes_also) else includes_also
excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
excludes = "" if pd.isna(excludes) else excludes

num_samples = 2
gold_standard = ["A fischeeeee", "A Weizeeeen"]
gold_standard = []

## Generate Few-Shot Data

In [52]:
# select gold standard data

# ds_2_desc  = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

# ds_2_desc  = ds_2_desc[pd.notna(ds_2_desc["Description"])]

# gold_standard = []
# # 1. take one of each lvl 3 class:
# for lvl_3 in ds_2_desc[pd.notna(ds_2_desc["Description"])].groupby("NACE_lvl_3").size().index: 
#     gold_standard.append(ds_2_desc[ds_2_desc["NACE_lvl_3"] == lvl_3].iloc[0])

# df_gold_standard = pd.concat(gold_standard, axis=1).T

#df_gold_standard.to_csv("/Users/hendrikweichel/Downloads/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv")

In [53]:
#df_gold_standard = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")
df_gold_standard = pd.read_csv("data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

### Hyperparams

In [54]:
prompt_path = "generate_synthetic_data/prompts/prompts_5.json"
print(get_system_prompt(prompt_path))
print()
print(get_few_shot_user_prompt(prompt_path))

You generate realistic paragraphs from corporate annual reports that are a presentation of the copmanies business model. The text must: - sound natural and specific - avoid textbook or definitional language - vary structure, length, and narrative style - include concrete operational details The text must NOT: - repeat industry definitions - follow a fixed template - explicitly explain what the company does in generic terms


TASK 
You get a short description of a companies' busines model and rephrase it such that it fits into a typical description within an annual report.

DEFINITION
{includes} {includes_also}

{excludes}

Here are some examples of descriptions of these classes: 
```
{gold_standard}
```

BUSINESS MODEL 
{business_model}

INSTRUCTIONS
 - Create a business model description that is in a similar format as the examples
 - Do NOT exactly copy phrases, sentence patterns, or structure from the examples
 - Think of the examples as constraints, not templates
 - Also include ter

In [55]:
level = 1
head_nace_code = "1" if level > 1 else None
generated_classes = nace_descriptions[nace_descriptions["PARENT_ID"] == head_nace_code]["CODE"]
few_shot = True

In [56]:
# generate date 

date = datetime.datetime.now().strftime("%Y%m%d")
date = "20251218"
suffix = "__few_shot" if few_shot else "__zero_shot"
store_path = "data/synthetic_data/data_" + date + f"__level_{level}__subclasses_{head_nace_code}__{os.path.basename(prompt_path).replace('.json', '')}{suffix}/" 
os.makedirs(store_path, exist_ok=True)

In [57]:
generated_data = {}

In [58]:
generated_classes = ["A", "B", "C", "J", "F"]
generated_classes = ["A", "C"]

In [59]:
# load previous results
prev_results = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_5__few_shot/class_A.csv"
prev_results = "data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_5__few_shot"
for class_name in generated_classes: 
   file_path = os.path.join(prev_results, f"class_{class_name}.csv")
   try:
       df = pd.read_csv(file_path)
       generated_data[class_name] = {
           "data": df[class_name].tolist()
       }
   except Exception as e:
       print(f"Could not load previous results for class {class_name}: {e}")

### Generate Topics

In [63]:
num_samples = 5
iterations_ = 20

In [64]:
topics = {}

for generate_nace_class in generated_classes[1:]:

    includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
    if pd.isna(includes):
        print("No description for class:", generate_nace_class)
        continue
    includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
    includes_also = "" if pd.isna(includes_also) else includes_also
    excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
    excludes = "" if pd.isna(excludes) else excludes

    subsections = get_sublevels(generate_nace_class, level=4)

    examples = []

    for i in tqdm(range(iterations_), desc=generate_nace_class):
        res = generate_topic(num_samples=num_samples, 
                             includes=includes, 
                             includes_also=includes_also, 
                             excludes=excludes, 
                             subsections=subsections, 
                             prompt_path=prompt_path, 
                             model="gpt-4o-mini", 
                             temperature=0.8)
        examples.append(res[1])
        data = split_synthetic_data("\n\n".join(examples), num_samples * iterations_)
        if len(data) >= num_samples * iterations_:
            break

    results = {
        "topics": data,
        "prompt": res[0],
        "system_prompt": res[0].messages[0].content,
        "user_prompt": res[0].messages[1].content,
        "output": examples,
        "few_shot": few_shot
    }

    topics[generate_nace_class] = results

C:   5%|████████▊                                                                                                                                                                      | 1/20 [00:10<03:24, 10.79s/it]


KeyboardInterrupt: 

In [66]:
print(res[0].messages[1].content)


TASK
For the following business sector, create short descriptions of realistic business models for exisiting companies. 


DEFINITION

This section includes the physical or chemical transformation of materials, substances, or components into new products, although this cannot be used as the single universal criterion for defining manufacturing (see remark on processing of waste below). The materials, substances, or components transformed are raw materials that are products of agriculture, forestry, fishing, mining or quarrying as well as products of other manufacturing activities. Substantial alteration, renovation or reconstruction of goods is generally considered to be manufacturing.\n\nThe output of a manufacturing process may be finished in the sense that it is ready for utilisation or consumption, or it may be semi-finished in the sense that it is to become an input for further manufacturing. For example, the output of alumina refining is the input used in the primary production 

In [95]:
for e in examples: 
    print("---")
    print(e)

---
1. **Processing and preserving of meat:** A company specializing in the production of premium, ready-to-eat jerky from locally sourced, grass-fed beef and poultry, targeting health-conscious consumers. 

2. **Manufacture of dairy products:** A dairy cooperative that produces organic cheese using traditional recipes and sustainable practices, distributing through local farmer's markets and upscale grocery stores.

3. **Manufacture of beverages:** A craft brewery creating small-batch, artisanal beers with unique flavor profiles, focusing on local ingredients and sustainable brewing methods.

4. **Manufacture of bakery and farinaceous products:** A bakery that offers gluten-free pastries and breads, catering to the growing demand for alternative dietary options in urban areas.

5. **Manufacture of grain mill products:** A company producing whole grain flours from heirloom grains, providing organic options to consumers and local bakeries through direct sales.

6. **Manufacture of prepa

### Generate Descriptions

In [134]:

#for generate_nace_class in generated_classes:
for generate_nace_class in ["A"]:

    includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
    if pd.isna(includes):
        print("No description for class:", generate_nace_class)
        continue
    includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
    includes_also = "" if pd.isna(includes_also) else includes_also
    excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
    excludes = "" if pd.isna(excludes) else excludes

    if few_shot: 
        gold_standard = df_gold_standard[df_gold_standard["NACE_letter"] == generate_nace_class]["Description_clean"].to_list()[:3] 
        gold_standard = [text.replace("\n", "") for text in gold_standard]
    else: 
        gold_standard = [] 

    subsections = get_sublevels(generate_nace_class, level=2)

    examples = []

    if generated_data.get(generate_nace_class) is not None:
        if len(generated_data[generate_nace_class].get("data", [])) > 0:
            examples = generated_data[generate_nace_class]["data"]

    for i in tqdm(range(iterations_*num_samples), desc=generate_nace_class):
        topic = topics[generate_nace_class]["topics"][i]
        res = generate_synthetic_data(num_samples=10, gold_standard=gold_standard, includes=includes, includes_also=includes_also, excludes=excludes, subsections=subsections, prompt_path=prompt_path, model="gpt-4o-mini", topic=topic)
        examples.append(res[1])
        data = split_synthetic_data("\n\n".join(examples), num_samples * iterations_)
        pd.DataFrame(data, columns=[generate_nace_class]).to_csv(os.path.join(store_path, f"class_{generate_nace_class}.csv"), index=False)
        if len(data) >= num_samples * iterations_:
            break

    results = {
        "data": data,
        "prompt": res[0],
        "system_prompt": res[0].messages[0].content,
        "user_prompt": res[0].messages[1].content,
        "output": examples, 
        "few_shot": few_shot
    }

    generated_data[generate_nace_class] = results

A:   0%|                                                                                                                                                                                     | 0/1000 [00:00<?, ?it/s]

input_variables=['business_model', 'excludes', 'gold_standard', 'includes', 'includes_also'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You generate realistic paragraphs from corporate annual reports that are a presentation of the copmanies business model. The text must: - sound natural and specific - avoid textbook or definitional language - vary structure, length, and narrative style - include concrete operational details The text must NOT: - repeat industry definitions - follow a fixed template - explicitly explain what the company does in generic terms'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['business_model', 'excludes', 'gold_standard', 'includes', 'includes_also'], input_types={}, partial_variables={}, template="\nTASK \nYou get a short description of a companies' busines model and rephrase it such that it fit

A:   0%|▏                                                                                                                                                                          | 1/1000 [00:04<1:15:38,  4.54s/it]

input_variables=['business_model', 'excludes', 'gold_standard', 'includes', 'includes_also'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You generate realistic paragraphs from corporate annual reports that are a presentation of the copmanies business model. The text must: - sound natural and specific - avoid textbook or definitional language - vary structure, length, and narrative style - include concrete operational details The text must NOT: - repeat industry definitions - follow a fixed template - explicitly explain what the company does in generic terms'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['business_model', 'excludes', 'gold_standard', 'includes', 'includes_also'], input_types={}, partial_variables={}, template="\nTASK \nYou get a short description of a companies' busines model and rephrase it such that it fit

A:   0%|▎                                                                                                                                                                          | 2/1000 [00:08<1:08:53,  4.14s/it]

...

input_variables=['business_model', 'excludes', 'gold_standard', 'includes', 'includes_also'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You generate realistic paragraphs from corporate annual reports that are a presentation of the copmanies business model. The text must: - sound natural and specific - avoid textbook or definitional language - vary structure, length, and narrative style - include concrete operational details The text must NOT: - repeat industry definitions - follow a fixed template - explicitly explain what the company does in generic terms'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['business_model', 'excludes', 'gold_standard', 'includes', 'includes_also'], input_types={}, partial_variables={}, template="\nTASK \nYou get a short description of a companies' busines model and rephrase it such that it fit

A:  52%|████████████████████████████████████████████████████████████████████████████████████████▌                                                                                  | 518/1000 [37:09<32:14,  4.01s/it]

input_variables=['business_model', 'excludes', 'gold_standard', 'includes', 'includes_also'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You generate realistic paragraphs from corporate annual reports that are a presentation of the copmanies business model. The text must: - sound natural and specific - avoid textbook or definitional language - vary structure, length, and narrative style - include concrete operational details The text must NOT: - repeat industry definitions - follow a fixed template - explicitly explain what the company does in generic terms'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['business_model', 'excludes', 'gold_standard', 'includes', 'includes_also'], input_types={}, partial_variables={}, template="\nTASK \nYou get a short description of a companies' busines model and rephrase it such that it fit

: 

: 

In [130]:
topics["C"]["topics"][1]

"2. **Manufacture of dairy products:** A dairy cooperative that produces organic cheese using traditional recipes and sustainable practices, distributing through local farmer's markets and upscale grocery stores."

In [119]:
# print prompts
for k,v in generated_data.items(): 
    print(k,len(v["output"]), "_______"*20)
    for k in v["output"]: 
        print(k)
        print("_")

A 20 ____________________________________________________________________________________________________________________________________________
The company operates a pioneering rice farm that champions sustainable agricultural practices, prioritizing eco-friendly methods to cultivate high-quality organic rice. With a strong commitment to environmental stewardship, we employ innovative techniques that enhance soil health and biodiversity, ensuring our farming operations have a minimal ecological footprint. Our rice is grown in harmony with nature, utilizing natural pest control and organic fertilizers that contribute to the overall vitality of the ecosystem.

In addition to our cultivation practices, we emphasize local distribution channels, connecting directly with health-conscious consumers who value transparency and quality in their food sources. By fostering relationships with local markets and retailers, we ensure that our organic rice reaches customers at peak freshness, reinfo

In [110]:
len(v["data"])

68

In [98]:
# print prompts
for k,v in generated_data.items(): 
    print(k,"_______"*20)
    print(v["user_prompt"])


A ____________________________________________________________________________________________________________________________________________

TASK 
You get a short description of a companies' busines model and rephrase it such that it fits into a typical description within an annual report.

DEFINITION
This section includes the exploitation of vegetal and animal natural resources, comprising the activities of growing of crops, raising and breeding of animals, harvesting of timber and other plants, animals or animal products from a farm or their natural habitats. 



Here are some examples of descriptions of these classes: 
```
Example 1:
The company, together with its subsidiaries, operates as one of the largest vertically integrated agricultural groups in Ukraine, engaging in the production, storage, processing, and sale of agricultural products. The company's key activities include breeding pigs, processing pork, and producing wheat and sunflower. The Group focuses on three winter 

#### aggregate data and split

In [26]:
generated_data["A"].keys()

dict_keys(['data'])

In [31]:
# config

config = {
    #
    # 
    "prompts": {k: v.get("user_prompt") for k, v in generated_data.items()}, 
    #"samples": num_samples * iterations_,
    #"generated_iterations": iterations_,
    "level": level,
    "head_nace_code": head_nace_code,
    "system_prompt": get_system_prompt(prompt_path), 
    "few_shot_prompting": few_shot
}

# store
import json
with open(os.path.join(store_path, "config.json"), "w") as f: 
    json.dump(config, f, indent=4)

In [32]:
df_full = []
for k, v in generated_data.items(): 
    df_temp = pd.DataFrame(v["data"], columns=["text"])
    df_temp["label"] = k
    df_full.append(df_temp)
df_full = pd.concat(df_full, axis=0)

In [33]:
df_full

,text,label
0,The company operates a pioneering rice farm th...,A
1,"In addition to our cultivation practices, we e...",A
2,The company has established itself as a pionee...,A
3,The company operates a family-owned dairy farm...,A
4,The company operates a comprehensive agrofores...,A
...,...,...
996,"In 2023, we expanded our operations by integra...",C
997,"Furthermore, we actively collaborate with arch...",C
998,"At EcoModular Homes, we are redefining the lan...",C
999,Our modular units are designed with flexibilit...,C


In [34]:
# make new index from 0 to len(df_full)-1
df_full = df_full.reset_index(drop=True)
df_full

,text,label
0,The company operates a pioneering rice farm th...,A
1,"In addition to our cultivation practices, we e...",A
2,The company has established itself as a pionee...,A
3,The company operates a family-owned dairy farm...,A
4,The company operates a comprehensive agrofores...,A
...,...,...
1997,"In 2023, we expanded our operations by integra...",C
1998,"Furthermore, we actively collaborate with arch...",C
1999,"At EcoModular Homes, we are redefining the lan...",C
2000,Our modular units are designed with flexibilit...,C


In [35]:
import re
clean_text = lambda x: re.sub(r'^\d+\.\s*', " ", x).strip()

In [36]:
df_full["text"] = df_full["text"].apply(clean_text)

In [37]:
df_full.to_csv(os.path.join(store_path, "synthetic_data_full.csv"), index=False)

In [38]:
# make train test split 6:2:2

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_full, test_size=0.4, random_state=42, stratify=df_full["label"])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

len(train_df), len(test_df), len(val_df)

(1201, 400, 401)

In [39]:
train_df.to_csv(os.path.join(store_path, "train_data.csv"), index=False)
test_df.to_csv(os.path.join(store_path, "test_data.csv"), index=False)
val_df.to_csv(os.path.join(store_path, "val_data.csv"), index=False)

In [ ]:
llm = ChatOllama(
            model="llama3.1:8b-instruct-fp16",
            temperature=0.1, 
            base_url="http://10.80.20.101:11434/"
        )

In [ ]:
llm.invoke("HI")

ResponseError: llama runner process has terminated: CUDA error: out of memory
  current device: 0, in function ggml_backend_cuda_device_get_memory at //ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:3238
  cudaMemGetInfo(free, total)
//ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:84: CUDA error (status code: 500)

In [ ]:
import tiktoken

tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")

In [ ]:
len(tokenizer.encode("""1. Our company specializes in sustainable logging practices, focusing on the selective harvesting of timber from both natural and planted forests. We employ advanced techniques to minimize environmental impact while maximizing yield. Our operations include the use of eco-friendly machinery that reduces soil disturbance and promotes forest regeneration. Additionally, we provide firewood and charcoal products sourced from responsibly managed forests, ensuring that our offerings meet high environmental standards. By collaborating with local communities, we also support the gathering of non-wood forest products, enhancing biodiversity and fostering economic development in rural areas.

2. In our forestry operations, we prioritize silviculture techniques that enhance forest health and productivity. We implement practices such as thinning and controlled burns to promote the growth of high-quality timber. Our team conducts regular assessments to monitor forest conditions and adapt our management strategies accordingly. We also engage in the collection of non-wood products like wild mushrooms and medicinal herbs, which are harvested sustainably to ensure long-term availability. By integrating these activities, we create a diversified revenue stream while contributing to the conservation of forest ecosystems.

3. Our logging company is committed to responsible forest management, focusing on the extraction of roundwood in a way that preserves the ecological balance of the forest. We utilize state-of-the-art equipment designed to minimize waste and ensure the efficient harvesting of timber. Our operations are complemented by a robust training program for our workforce, emphasizing safety and environmental stewardship. Furthermore, we offer a range of products, including pit-props and pulpwood, which are supplied to various industries, ensuring that our timber is used effectively and sustainably.

4. We are dedicated to the gathering of wild growing non-wood forest products, which play a vital role in supporting local economies and promoting biodiversity. Our team works closely with foragers to identify and harvest edible plants, berries, and nuts in a sustainable manner. By implementing strict guidelines on harvesting practices, we ensure that these resources are available for future generations. Additionally, we provide training and resources to local communities, empowering them to participate in this industry while preserving their traditional knowledge and practices.

5. Our company offers comprehensive support services to forestry operations, including consulting on sustainable practices and forest management planning. We provide expertise in areas such as reforestation, pest management, and soil conservation, helping clients optimize their forestry activities. Our services extend to training programs for forest workers, focusing on safety protocols and sustainable harvesting techniques. By partnering with clients, we aim to enhance the productivity and sustainability of their forestry operations, contributing to the overall health of forest ecosystems.

6. We focus on the extraction of high-quality roundwood, utilizing advanced logging techniques that prioritize sustainability and efficiency. Our operations are designed to minimize waste and enhance the recovery of valuable timber products. We also engage in the production of firewood, which is sourced from our managed forests and processed to meet consumer demand. Our commitment to responsible forestry practices ensures that we not only meet market needs but also contribute positively to the environment and local communities.

7. Our forestry management firm specializes in the cultivation and maintenance of planted forests, ensuring a steady supply of timber for various applications. We implement innovative silvicultural practices that enhance growth rates and timber quality, while also focusing on biodiversity conservation. In addition to timber production, we engage in the collection of non-wood forest products, such as wild herbs and berries, which are marketed to local businesses. This dual approach allows us to maximize the economic value of our forests while promoting ecological health.

8. We are involved in the logging sector, where our operations emphasize the careful extraction of timber from both natural and managed forests. Our commitment to sustainability is reflected in our use of low-impact logging techniques, which help preserve the integrity of the forest ecosystem. We also produce charcoal and firewood, catering to the growing demand for renewable energy sources. By maintaining a focus on environmental responsibility, we strive to balance economic viability with ecological preservation.

9. Our company is dedicated to the sustainable gathering of wild growing non-wood products, which are integral to the livelihoods of many local communities. We prioritize ethical harvesting practices that ensure the long-term availability of these resources. Our team collaborates with local foragers to promote best practices and provide training on sustainable collection techniques. By creating a market for these products, we not only support local economies but also contribute to the conservation of forest biodiversity.

10. We provide essential support services to forestry operations, helping clients navigate the complexities of sustainable forest management. Our team offers expertise in areas such as land assessment, resource inventory, and compliance with environmental regulations. We also facilitate training programs focused on best practices for logging and non-wood product harvesting. By equipping forestry businesses with the knowledge and tools they need, we aim to enhance their operational efficiency and promote sustainable practices across the sector.

"""))

In [ ]:
pd.DataFrame(data, columns=[generate_nace_class])

In [ ]:
data